# 01 — Learn-Then-Test (LTT) for Risk Control

This notebook illustrates a simple **Learn-Then-Test (LTT)** procedure for
hyperparameter selection with **finite-sample risk control** on the **average loss**.

We will:

1. Generate a synthetic CSV file `../data/sample_LTT_losses.csv` if it does not exist.
2. Explain its format (multiple loss values per hyperparameter).
3. Implement a basic LTT pipeline:
   - Compute empirical average risk for each hyperparameter.
   - Convert these to p-values using a Hoeffding-style bound.
   - Apply the Bonferroni correction as a simple FWER-controlling method.
4. Inspect which hyperparameters are declared *reliable* at level `α, δ`.


In [ ]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Make plots inline if in classic Notebook
# %matplotlib inline  # Uncomment if needed

BASE_DIR = Path("..").resolve()
DATA_DIR = BASE_DIR / "data"
DATA_DIR.mkdir(parents=True, exist_ok=True)

LTT_CSV = DATA_DIR / "sample_LTT_losses.csv"

print("Base directory:", BASE_DIR)
print("Data directory:", DATA_DIR)
print("LTT CSV path:", LTT_CSV)


In [ ]:
def generate_sample_ltt_csv(path: Path, num_lambdas: int = 20, num_calib: int = 100, seed: int = 0):
    """Generate a synthetic CSV with multiple calibration losses per hyperparameter.

    Format:
    - One row per hyperparameter configuration λ.
    - Columns:
        - 'lambda_id' (int)
        - 'loss_1', 'loss_2', ..., 'loss_M' (floats in [0, 1]).
    """
    rng = np.random.default_rng(seed)
    lambda_ids = np.arange(num_lambdas)

    # For illustration, we create a few "good" configs and a few "bad" configs
    # Good configs have lower mean loss; bad ones have higher mean loss.
    base_means = np.linspace(0.05, 0.35, num_lambdas)  # target mean losses per λ
    losses = []
    for mu in base_means:
        # Truncate to [0,1] just for nice plots
        row = np.clip(rng.normal(loc=mu, scale=0.05, size=num_calib), 0.0, 1.0)
        losses.append(row)

    losses = np.stack(losses, axis=0)  # shape (num_lambdas, num_calib)

    cols = ["lambda_id"] + [f"loss_{i+1}" for i in range(num_calib)]
    data = np.column_stack([lambda_ids, losses])
    df = pd.DataFrame(data, columns=cols)
    df["lambda_id"] = df["lambda_id"].astype(int)

    df.to_csv(path, index=False)
    return df


if not LTT_CSV.exists():
    print("CSV does not exist — generating synthetic sample_LTT_losses.csv ...")
    df_ltt = generate_sample_ltt_csv(LTT_CSV)
else:
    print("CSV already exists — loading:", LTT_CSV)
    df_ltt = pd.read_csv(LTT_CSV)

print("\nFirst few rows of the LTT CSV:")
display(df_ltt.head())

print("\nColumns:")
print(df_ltt.columns.tolist())


In [ ]:
# Extract loss matrix
lambda_ids = df_ltt["lambda_id"].values
loss_cols = [c for c in df_ltt.columns if c.startswith("loss_")]
loss_mat = df_ltt[loss_cols].values  # shape (num_lambdas, num_calib)

num_lambdas, num_calib = loss_mat.shape
print(f"Number of hyperparameters: {num_lambdas}")
print(f"Number of calibration losses per hyperparameter: {num_calib}")

# Empirical average risk for each λ
R_hat = loss_mat.mean(axis=1)

plt.figure()
plt.plot(lambda_ids, R_hat, "o-")
plt.xlabel("lambda_id")
plt.ylabel("Empirical average loss R_hat(λ)")
plt.title("Per-hyperparameter empirical average loss")
plt.grid(True)
plt.show()


In [ ]:
# === LTT: Hoeffding p-values + Bonferroni ===

def hoeffding_p_value(r_hat, alpha, n):
    """One-sided Hoeffding-style p-value for H: R(λ) > alpha.

    Under the null (true risk > alpha), the probability that the empirical mean
    is this low or lower is bounded by the Hoeffding exponent.
    """
    # If empirical risk is already above alpha, then p-value = 1 is safe
    if r_hat >= alpha:
        return 1.0
    return float(np.exp(-2.0 * n * (alpha - r_hat) ** 2))


alpha = 0.2  # risk level to control (e.g., max acceptable average loss)
delta = 0.1  # family-wise error tolerance

p_values = np.array([hoeffding_p_value(r, alpha, num_calib) for r in R_hat])

# Bonferroni correction
bonf_threshold = delta / num_lambdas

rejected = p_values < bonf_threshold
selected_lambdas = lambda_ids[rejected]

results_df = pd.DataFrame({
    "lambda_id": lambda_ids,
    "R_hat": R_hat,
    "p_value": p_values,
    "selected_by_Bonferroni": rejected,
}).sort_values("R_hat")

print(f"Alpha (risk level): {alpha}")
print(f"Delta (FWER target): {delta}")
print(f"Bonferroni threshold: {bonf_threshold:.4g}\n")

print("Hyperparameters sorted by empirical risk:")
display(results_df.head(10))

print("\nSelected hyperparameters (Bonferroni):", selected_lambdas.tolist())


In [ ]:
# Optional: simple Fixed-Sequence Testing (FST) based on ordering by empirical risk.

order = np.argsort(R_hat)  # from lowest to highest empirical risk
fst_selected = []
for idx in order:
    p = p_values[idx]
    if p < delta:
        fst_selected.append(lambda_ids[idx])
    else:
        # stop at first non-rejection
        break

print("Fixed-Sequence Testing (FST) selection (order by R_hat):", fst_selected)
